---
title: Escenarios de riesgo ante el Fenómeno El Niño 2026–2027
subtitle: Análisis del análogo histórico Niño 2023-2024 para la anticipación de emergencias en Colombia
author:
  - name: Christian Euscátegui
    affiliations: Subdirección para el Conocimiento del Riesgo (SCR), UNGRD
license: CC-BY-4.0
date: 2026-05-20
keywords:
  - El Niño
  - Fenómeno El Niño
  - Riesgo de desastres
  - Colombia
  - Análisis espacial
  - Análogo climático
  - UNGRD
abbreviations:
  UNGRD: Unidad Nacional para la Gestión del Riesgo de Desastres
  SCR: Subdirección para el Conocimiento del Riesgo
  IDEAM: Instituto de Hidrología, Meteorología y Estudios Ambientales
  DIVIPOLA: División Político-Administrativa de Colombia
  DANE: Departamento Administrativo Nacional de Estadística
---

+++ {"part": "abstract"}

Este cuaderno analiza los patrones históricos de emergencias registradas por la UNGRD durante el Fenómeno El Niño 2023–2024, utilizándolo como análogo para anticipar los posibles escenarios de riesgo ante un eventual Fenómeno El Niño 2026–2027 en Colombia. Se procesan registros del periodo junio 2023–mayo 2024, clasificando los eventos por categoría climática (condiciones lluviosas y secas), tipo de amenaza y dimensión territorial (departamental y municipal). Los resultados se presentan mediante mapas coropléticos, gráficos de barras y tablas de indicadores de impacto orientados a la toma de decisiones en gestión del riesgo de desastres.

## Contexto y objetivo

El Fenómeno El Niño es una de las perturbaciones climáticas de mayor impacto sobre el territorio colombiano. Sus efectos se manifiestan de forma diferenciada: en gran parte del país genera déficit hídrico y condiciones de sequía que favorecen incendios de la cobertura vegetal, desabastecimiento de agua y heladas; mientras que en otras regiones puede intensificar las lluvias, incrementando el riesgo de inundaciones, movimientos en masa y avenidas torrenciales.

Ante las proyecciones del IDEAM y de organismos internacionales que señalan una probabilidad significativa de que se configure un nuevo Fenómeno El Niño durante 2026–2027, la Subdirección para el Conocimiento del Riesgo de la UNGRD desarrolla el presente análisis de escenarios basado en el método del análogo histórico: se toman los registros de emergencias del Niño 2023–2024 —el evento de referencia más reciente— para identificar los patrones territoriales, temporales y por tipo de evento que probablemente se repetirían bajo condiciones climáticas similares.

**Objetivo principal:** Identificar los departamentos, municipios, tipos de evento y periodos del año con mayor probabilidad de concentración de emergencias durante un eventual Fenómeno El Niño 2026–2027, con base en la evidencia del análogo 2023–2024.

**Objetivos específicos:**
- Cuantificar el impacto total (frecuencia, familias, personas, infraestructura) del análogo Niño 2023–2024.
- Clasificar los eventos según la condición climática que los favorece (lluviosa o seca).
- Generar rankings territoriales por métrica de impacto.
- Visualizar la distribución temporal y espacial de los eventos.

:::{note}
Este análisis no constituye una predicción determinística. Los resultados representan escenarios de riesgo *plausibles* derivados del comportamiento histórico. La materialización efectiva de las emergencias dependerá de la intensidad y duración del eventual fenómeno climático, las acciones de reducción del riesgo implementadas y las condiciones de exposición y vulnerabilidad de cada territorio.
:::

## Datos de entrada

| Fuente | Archivo | Descripción |
|--------|---------|-------------|
| UNGRD / SCR | `datos_base_analisis.parquet` | Registros de emergencias 2023–2024 con variables de impacto, tipo de evento, clasificación climática y codificación DIVIPOLA |
| DANE / IGAC | `colombia_departamentos.json` | Geometrías departamentales de Colombia en formato GeoJSON (DIVIPOLA) |
| DANE / IGAC | `colombia_municipios.json` | Geometrías municipales de Colombia en formato GeoJSON (DIVIPOLA) |

Los archivos deben ubicarse en una carpeta `data/` en el mismo directorio del cuaderno. Consulte el archivo `data/README.md` para instrucciones de descarga.

**Periodo de análisis:** Junio 2023 – Mayo 2024  
**Resolución espacial:** Departamental y municipal (DIVIPOLA)  
**Variables de impacto analizadas:** Frecuencia de eventos, familias afectadas, personas afectadas, acueductos afectados, centros de salud afectados, vías afectadas, hectáreas afectadas.

**Clasificación climática de eventos:**
- 🌧️ **Condiciones lluviosas:** Avenida torrencial, creciente súbita, granizada, inundación, movimiento en masa, tormenta eléctrica, vendaval.
- ☀️ **Condiciones secas:** Desabastecimiento de agua, helada, incendio forestal, racionamiento, sequía.

## Metodología

El flujo de trabajo comprende cuatro etapas: (1) carga y preprocesamiento de los registros de emergencias, (2) clasificación climática y agregación por categoría, (3) construcción de indicadores de impacto y rankings territoriales, y (4) generación de visualizaciones cartográficas y estadísticas.

Para la visualización geográfica se emplea el método de coropléticos con clasificación por cuartiles (`quantile`), que es más robusto ante distribuciones sesgadas —habituales en datos de emergencias— que la clasificación por intervalos iguales (`equal interval`). Los mapas usan la proyección implícita de los GeoJSON de origen (WGS84 / EPSG:4326).

### Configuración del entorno

In [ ]:
import pandas as pd
import json
import os
import warnings

import altair as alt
import folium
from IPython.display import display, HTML

warnings.filterwarnings('ignore')
alt.data_transformers.disable_max_rows()

# Paleta institucional UNGRD
AZUL_UNGRD  = '#0056b3'
ROJO_UNGRD  = '#dc3545'
AZUL_OSCURO = '#003366'
GRIS_FONDO  = '#f8f9fa'

### Rutas y parámetros

In [ ]:
RUTA_DATOS      = os.path.join('data', 'datos_base_analisis.parquet')
RUTA_DEPTOS     = os.path.join('data', 'colombia_departamentos.json')
RUTA_MUNICIPIOS = os.path.join('data', 'colombia_municipios.json')

# Orden cronológico de trimestres del análogo
ORDEN_TRIMESTRES = [
    'junio-agosto 2023',
    'septiembre-noviembre 2023',
    'diciembre 2023–febrero 2024',
    'marzo-mayo 2024'
]

# Mapa de códigos de trimestre a etiquetas legibles
TRIM_MAP = {
    'JJA_2023':       'junio-agosto 2023',
    'SON_2023':       'septiembre-noviembre 2023',
    'DEF_2023_2024':  'diciembre 2023–febrero 2024',
    'MAM_2024':       'marzo-mayo 2024'
}

# Variables de impacto disponibles
METRICAS = {
    'Frecuencia (Total eventos)':    'FRECUENCIA',
    'Familias afectadas':            'FAMILIAS',
    'Personas afectadas':            'PERSONAS',
    'Acueductos afectados':          'ACUED.',
    'Centros de salud afectados':    'C.SALUD',
    'Vías afectadas':                'VIAS',
    'Hectáreas afectadas':           'HECTAREAS'
}

### Carga y preprocesamiento de datos

In [ ]:
def cargar_base_maestra(ruta: str) -> pd.DataFrame:
    """Carga y filtra los registros del análogo Niño 2023-2024."""
    df = pd.read_parquet(ruta)
    df = df[df['FECHA_DATETIME'] >= pd.to_datetime('2023-06-01')].copy()
    df['TRIMESTRE_FORMAL'] = df['PERIODO_TRIMESTRAL'].map(TRIM_MAP)
    return df


def cargar_geojsons(ruta_deptos: str, ruta_municipios: str) -> tuple:
    """Carga los GeoJSON y normaliza las claves COD_GEO y NOMBRE_GEO."""
    with open(ruta_deptos, 'r', encoding='utf-8') as f:
        geo_d = json.load(f)
    for feat in geo_d.get('features', []):
        feat['properties']['COD_GEO']    = str(feat['properties'].get('DPTO_CCDGO', '0')).zfill(2)
        feat['properties']['NOMBRE_GEO'] = feat['properties'].get('DPTO_CNMBR', 'Desconocido')

    with open(ruta_municipios, 'r', encoding='utf-8') as f:
        geo_m = json.load(f)
    for feat in geo_m.get('features', []):
        cod = feat['properties'].get('MPIO_CCNCT') or feat['properties'].get('DPMP') or '0'
        feat['properties']['COD_GEO']    = str(cod).zfill(5)
        feat['properties']['NOMBRE_GEO'] = feat['properties'].get('MPIO_CNMBR', 'Desconocido')

    return geo_d, geo_m


df = cargar_base_maestra(RUTA_DATOS)
geo_deptos, geo_municipios = cargar_geojsons(RUTA_DEPTOS, RUTA_MUNICIPIOS)

print(f'Registros cargados:  {len(df):,}')
print(f'Periodo:             {df["FECHA_DATETIME"].min().date()}  →  {df["FECHA_DATETIME"].max().date()}')
print(f'Departamentos:       {df["Nombre Departamento"].nunique()}')
print(f'Municipios:          {df["Nombre Municipio"].nunique()}')
print(f'Tipos de evento:     {df["EVENTO"].nunique()}')

## Resultados e interpretación

### Indicadores de impacto globales

La siguiente tabla resume los indicadores de impacto agregados para todo el periodo del análogo (junio 2023 – mayo 2024). Estos valores constituyen el límite superior de referencia para los escenarios de riesgo ante un Niño 2026–2027 de magnitud e intensidad comparable.

In [ ]:
resumen = {
    'Total de eventos':              len(df),
    'Familias afectadas':            int(df['FAMILIAS'].sum()),
    'Personas afectadas':            int(df['PERSONAS'].sum()),
    'Acueductos afectados':          int(df['ACUED.'].sum()),
    'Centros de salud afectados':    int(df['C.SALUD'].sum()),
    'Vías afectadas':                int(df['VIAS'].sum()),
    'Hectáreas afectadas':           int(df['HECTAREAS'].sum()),
}

df_kpi = pd.DataFrame(list(resumen.items()), columns=['Indicador', 'Valor'])

(
    df_kpi.style
    .format({'Valor': '{:,.0f}'})
    .set_caption('Indicadores de impacto — Análogo El Niño 2023–2024')
    .set_table_styles([
        {'selector': 'caption',
         'props': [('font-weight', 'bold'), ('color', AZUL_UNGRD),
                   ('font-size', '14px'), ('text-align', 'left')]},
        {'selector': 'th',
         'props': [('background-color', AZUL_UNGRD), ('color', 'white'),
                   ('font-size', '12px')]},
        {'selector': 'td:nth-child(2)',
         'props': [('text-align', 'right'), ('font-weight', 'bold')]},
    ])
    .hide(axis='index')
)

### Distribución por categoría climática y trimestre

La clasificación climática de los eventos permite separar los patrones propios del Fenómeno El Niño (condiciones secas) de los eventos que ocurren a pesar de la reducción de lluvias o que corresponden a las fases de transición del fenómeno. El gráfico muestra la evolución trimestral de ambas categorías.

In [ ]:
df_clima = (
    df.groupby(['TRIMESTRE_FORMAL', 'CATEGORIA_CLIMA'])
    .size()
    .reset_index(name='Eventos')
    .dropna(subset=['TRIMESTRE_FORMAL'])
)

chart_clima = (
    alt.Chart(df_clima)
    .mark_bar(cornerRadiusEnd=4)
    .encode(
        x=alt.X('TRIMESTRE_FORMAL:N', title='Trimestre',
                sort=ORDEN_TRIMESTRES,
                axis=alt.Axis(labelAngle=-25)),
        y=alt.Y('Eventos:Q', title='Total de eventos', stack=True),
        color=alt.Color(
            'CATEGORIA_CLIMA:N',
            scale=alt.Scale(
                domain=['Lluviosos', 'Secos'],
                range=[AZUL_UNGRD, ROJO_UNGRD]
            ),
            legend=alt.Legend(title='Categoría climática')
        ),
        tooltip=[
            alt.Tooltip('TRIMESTRE_FORMAL:N', title='Trimestre'),
            alt.Tooltip('CATEGORIA_CLIMA:N',  title='Categoría'),
            alt.Tooltip('Eventos:Q',           title='Eventos', format=',')
        ]
    )
    .properties(
        height=320,
        title=alt.TitleParams(
            text='Distribución trimestral de eventos por categoría climática',
            subtitle='Análogo El Niño 2023–2024'
        )
    )
    .configure_axis(labelColor='black', titleColor='black', tickColor='black', domainColor='black')
    .configure_title(color='black')
    .configure_legend(labelColor='black', titleColor='black')
)

chart_clima

### Distribución geográfica — Mapa coropléticO nacional

Los mapas coropléticos muestran la concentración espacial de los eventos y sus impactos a nivel departamental. La clasificación por cuartiles (Q1–Q4) facilita identificar los departamentos en el 25 % superior de la distribución, que son los territorios con mayor exposición histórica y donde la anticipación de acciones de gestión del riesgo es más urgente.

#### Frecuencia total de eventos

In [ ]:
def crear_mapa_coroplético(
    df_filtrado: pd.DataFrame,
    geo_data: dict,
    col_agrup: str,
    metrica: str,
    escala_color: str,
    titulo_leyenda: str,
    alias_geo: str = 'Territorio:',
    zoom_start: int = 5,
    fit_bounds: bool = False
) -> folium.Map:
    """Genera un mapa coropléticO interactivo con clasificación por cuartiles."""
    if metrica == 'FRECUENCIA':
        agrup = df_filtrado.groupby(col_agrup).size().reset_index(name='Valor')
    else:
        agrup = df_filtrado.groupby(col_agrup)[metrica].sum().reset_index(name='Valor')

    n_digits = 2 if col_agrup == 'Codigo Departamento' else 5
    agrup[col_agrup] = agrup[col_agrup].astype(str).str.zfill(n_digits)

    m = folium.Map(
        location=[4.5709, -74.2973],
        zoom_start=zoom_start,
        tiles='CartoDB positron',
        control_scale=True
    )

    # Ajuste de leyenda para mejor legibilidad
    css_leyenda = """
    <style>
        svg text { fill: #000000 !important; font-weight: bold !important; font-size: 10px !important; }
        .legend { transform: scale(0.88); transform-origin: top right;
                  background-color: rgba(255,255,255,0.85); border-radius: 4px; }
    </style>
    """
    m.get_root().html.add_child(folium.Element(css_leyenda))

    if agrup['Valor'].sum() == 0:
        return m

    calc_bins = agrup['Valor'].quantile([0, 0.25, 0.5, 0.75, 1.0]).tolist()
    u_bins    = sorted(set(calc_bins))
    n_bins    = u_bins if len(u_bins) >= 4 else 4

    cp = folium.Choropleth(
        geo_data=geo_data,
        name='choropleth',
        data=agrup,
        columns=[col_agrup, 'Valor'],
        key_on='feature.properties.COD_GEO',
        fill_color=escala_color,
        fill_opacity=0.80,
        line_opacity=0.30,
        legend_name=titulo_leyenda,
        bins=n_bins
    ).add_to(m)

    # Agrega valores al tooltip
    for s in cp.geojson.data['features']:
        val = agrup.loc[agrup[col_agrup] == s['properties']['COD_GEO'], 'Valor'].values
        s['properties']['Valor'] = int(val[0]) if len(val) > 0 else 0

    estilo_tooltip = (
        'font-size:11px; font-family:sans-serif; padding:5px; '
        'background:rgba(255,255,255,0.95); border:1px solid #0056b3; '
        'border-radius:4px; color:black;'
    )
    folium.GeoJsonTooltip(
        fields=['NOMBRE_GEO', 'Valor'],
        aliases=[alias_geo, f'{titulo_leyenda}:'],
        style=estilo_tooltip
    ).add_to(cp.geojson)

    if fit_bounds:
        m.fit_bounds(cp.get_bounds())

    return m


mapa_frecuencia = crear_mapa_coroplético(
    df_filtrado=df,
    geo_data=geo_deptos,
    col_agrup='Codigo Departamento',
    metrica='FRECUENCIA',
    escala_color='YlOrRd',
    titulo_leyenda='Frecuencia de eventos',
    alias_geo='Departamento:'
)
display(mapa_frecuencia)

#### Familias afectadas

La métrica de familias afectadas es uno de los indicadores de impacto humanitario más relevantes en la gestión del riesgo, ya que refleja directamente la dimensión de las emergencias sobre la población.

In [ ]:
mapa_familias = crear_mapa_coroplético(
    df_filtrado=df,
    geo_data=geo_deptos,
    col_agrup='Codigo Departamento',
    metrica='FAMILIAS',
    escala_color='YlOrBr',
    titulo_leyenda='Familias afectadas',
    alias_geo='Departamento:'
)
display(mapa_familias)

#### Eventos en condiciones secas (señal directa del Fenómeno El Niño)

Los eventos bajo condiciones secas son la señal más directa del impacto del Fenómeno El Niño. Incendios forestales, sequías, heladas y desabastecimiento de agua son los tipos de evento cuya concentración temporal se intensifica durante la fase cálida del ENSO.

In [ ]:
df_secos = df[df['CATEGORIA_CLIMA'] == 'Secos'].copy()

mapa_secos = crear_mapa_coroplético(
    df_filtrado=df_secos,
    geo_data=geo_deptos,
    col_agrup='Codigo Departamento',
    metrica='FRECUENCIA',
    escala_color='OrRd',
    titulo_leyenda='Eventos en condiciones secas',
    alias_geo='Departamento:'
)
display(mapa_secos)

### Ranking departamental

Los gráficos de barras presentan el Top 10 departamentos por cada variable de impacto. Estos rankings son la base para la priorización territorial de acciones de reducción del riesgo y preparación para la respuesta.

In [ ]:
def grafico_top10(
    df_entrada: pd.DataFrame,
    col_nombre: str,
    metrica: str,
    titulo: str,
    esquema_color: str = 'yelloworangered'
) -> alt.Chart:
    """Genera un gráfico de barras horizontales con el top 10 de territorios."""
    if metrica == 'FRECUENCIA':
        df_top = df_entrada[col_nombre].value_counts().reset_index()
        df_top.columns = ['Territorio', 'Valor']
    else:
        df_top = df_entrada.groupby(col_nombre)[metrica].sum().reset_index()
        df_top.columns = ['Territorio', 'Valor']

    df_top = df_top[df_top['Valor'] > 0].sort_values('Valor', ascending=False).head(10)

    return (
        alt.Chart(df_top)
        .mark_bar(cornerRadiusEnd=4)
        .encode(
            x=alt.X('Valor:Q', title='Valor'),
            y=alt.Y('Territorio:N', sort='-x', title='',
                    axis=alt.Axis(labelLimit=500, labelFontSize=11)),
            color=alt.Color('Valor:Q',
                            scale=alt.Scale(scheme=esquema_color),
                            legend=None),
            tooltip=[
                alt.Tooltip('Territorio:N', title='Departamento'),
                alt.Tooltip('Valor:Q',      title='Valor', format=',')
            ]
        )
        .properties(
            height=300,
            title=alt.TitleParams(text=titulo)
        )
        .configure_axis(
            labelColor='black', titleColor='black',
            tickColor='black',  domainColor='black'
        )
        .configure_title(color='black')
    )


# Frecuencia
grafico_top10(
    df, 'Nombre Departamento', 'FRECUENCIA',
    'Top 10 departamentos — Frecuencia de eventos',
    'yelloworangered'
)

In [ ]:
# Familias afectadas
grafico_top10(
    df, 'Nombre Departamento', 'FAMILIAS',
    'Top 10 departamentos — Familias afectadas',
    'yelloworangebrown'
)

In [ ]:
# Eventos en condiciones secas
grafico_top10(
    df_secos, 'Nombre Departamento', 'FRECUENCIA',
    'Top 10 departamentos — Eventos en condiciones secas',
    'orangered'
)

### Análisis temporal

La evolución trimestral de los eventos permite identificar los periodos de mayor concentración de emergencias. Esta información es crucial para planificar el preposicionamiento de ayudas humanitarias y el despliegue de capacidades de respuesta.

In [ ]:
df_trim = (
    df.groupby('TRIMESTRE_FORMAL')
    .size()
    .reset_index(name='Eventos')
    .dropna(subset=['TRIMESTRE_FORMAL'])
)

chart_temporal = (
    alt.Chart(df_trim)
    .mark_bar(cornerRadiusEnd=4, color=AZUL_UNGRD)
    .encode(
        x=alt.X('TRIMESTRE_FORMAL:N', title='Trimestre',
                sort=ORDEN_TRIMESTRES,
                axis=alt.Axis(labelAngle=-25)),
        y=alt.Y('Eventos:Q', title='Total de eventos'),
        tooltip=[
            alt.Tooltip('TRIMESTRE_FORMAL:N', title='Trimestre'),
            alt.Tooltip('Eventos:Q',           title='Eventos', format=',')
        ]
    )
    .properties(
        height=300,
        title=alt.TitleParams(
            text='Distribución trimestral de eventos',
            subtitle='Análogo El Niño 2023–2024 — Todos los tipos de evento'
        )
    )
    .configure_axis(
        labelColor='black', titleColor='black',
        tickColor='black',  domainColor='black'
    )
    .configure_title(color='black')
)

chart_temporal

### Análisis por tipo de evento

La desagregación por tipo de evento permite identificar cuáles amenazas concentran la mayor parte de las emergencias durante un episodio El Niño. Esta información orienta la formulación de planes de contingencia específicos por amenaza.

In [ ]:
df_tipo_total = df['EVENTO'].value_counts().reset_index()
df_tipo_total.columns = ['Tipo de evento', 'Frecuencia']

chart_tipo = (
    alt.Chart(df_tipo_total.head(15))
    .mark_bar(cornerRadiusEnd=4)
    .encode(
        x=alt.X('Frecuencia:Q', title='Número de eventos'),
        y=alt.Y('Tipo de evento:N', sort='-x', title='',
                axis=alt.Axis(labelLimit=500, labelFontSize=11)),
        color=alt.Color('Frecuencia:Q',
                        scale=alt.Scale(scheme='yelloworangebrown'),
                        legend=None),
        tooltip=[
            alt.Tooltip('Tipo de evento:N', title='Tipo'),
            alt.Tooltip('Frecuencia:Q',     title='Eventos', format=',')
        ]
    )
    .properties(
        height=420,
        title=alt.TitleParams(
            text='Top 15 tipos de evento — Análogo El Niño 2023–2024',
            subtitle='Frecuencia total del periodo junio 2023–mayo 2024'
        )
    )
    .configure_axis(
        labelColor='black', titleColor='black',
        tickColor='black',  domainColor='black'
    )
    .configure_title(color='black')
)

chart_tipo

#### Tipos de evento por categoría climática

In [ ]:
df_tipo_clima = (
    df.groupby(['EVENTO', 'CATEGORIA_CLIMA'])
    .size()
    .reset_index(name='Frecuencia')
)

# Top 12 eventos para cada categoría climática
top_eventos = df_tipo_total.head(12)['Tipo de evento'].tolist()
df_tipo_clima_top = df_tipo_clima[df_tipo_clima['EVENTO'].isin(top_eventos)]

chart_tipo_clima = (
    alt.Chart(df_tipo_clima_top)
    .mark_bar(cornerRadiusEnd=4)
    .encode(
        x=alt.X('Frecuencia:Q', title='Número de eventos'),
        y=alt.Y('EVENTO:N', sort='-x', title='',
                axis=alt.Axis(labelLimit=500, labelFontSize=11)),
        color=alt.Color(
            'CATEGORIA_CLIMA:N',
            scale=alt.Scale(
                domain=['Lluviosos', 'Secos'],
                range=[AZUL_UNGRD, ROJO_UNGRD]
            ),
            legend=alt.Legend(title='Categoría climática')
        ),
        tooltip=[
            alt.Tooltip('EVENTO:N',          title='Tipo de evento'),
            alt.Tooltip('CATEGORIA_CLIMA:N', title='Categoría'),
            alt.Tooltip('Frecuencia:Q',      title='Eventos', format=',')
        ]
    )
    .properties(
        height=380,
        title=alt.TitleParams(
            text='Tipos de evento por categoría climática',
            subtitle='Top 12 eventos — Análogo El Niño 2023–2024'
        )
    )
    .configure_axis(
        labelColor='black', titleColor='black',
        tickColor='black',  domainColor='black'
    )
    .configure_title(color='black')
    .configure_legend(labelColor='black', titleColor='black')
)

chart_tipo_clima

### Exportación de tablas de resultados

Las tablas de resultados se exportan en formato CSV para facilitar su uso en otros análisis o sistemas de información territorial.

In [ ]:
os.makedirs('resultados', exist_ok=True)

# Ranking departamental por frecuencia
ranking_depto = (
    df.groupby(['Codigo Departamento', 'Nombre Departamento'])
    .agg(
        Frecuencia   =('EVENTO',    'count'),
        Familias     =('FAMILIAS',  'sum'),
        Personas     =('PERSONAS',  'sum'),
        Acueductos   =('ACUED.',    'sum'),
        CentrosSalud =('C.SALUD',   'sum'),
        Vias         =('VIAS',      'sum'),
        Hectareas    =('HECTAREAS', 'sum')
    )
    .reset_index()
    .sort_values('Frecuencia', ascending=False)
)
ranking_depto.to_csv('resultados/ranking_departamentos.csv', index=False, encoding='utf-8-sig')

# Ranking municipal por frecuencia
ranking_mpio = (
    df.groupby(['Codigo Municipio', 'Nombre Municipio', 'Nombre Departamento'])
    .agg(
        Frecuencia   =('EVENTO',    'count'),
        Familias     =('FAMILIAS',  'sum'),
        Personas     =('PERSONAS',  'sum'),
    )
    .reset_index()
    .sort_values('Frecuencia', ascending=False)
)
ranking_mpio.to_csv('resultados/ranking_municipios.csv', index=False, encoding='utf-8-sig')

# Resumen por trimestre y categoría climática
resumen_trim = (
    df.groupby(['TRIMESTRE_FORMAL', 'CATEGORIA_CLIMA'])
    .agg(
        Frecuencia=('EVENTO', 'count'),
        Familias  =('FAMILIAS', 'sum'),
        Personas  =('PERSONAS', 'sum')
    )
    .reset_index()
    .dropna(subset=['TRIMESTRE_FORMAL'])
)
resumen_trim.to_csv('resultados/resumen_trimestral.csv', index=False, encoding='utf-8-sig')

print('Archivos exportados en la carpeta resultados/:')
for f in os.listdir('resultados'):
    print(f'  - {f}')

## Limitaciones y recomendaciones

**Limitaciones del análisis:**

1. **Subregistro de eventos:** Los datos de la UNGRD capturan emergencias que superan la capacidad de respuesta local y son reportadas a la red de gestión del riesgo. Eventos de menor magnitud pueden no estar registrados, especialmente en municipios con baja capacidad institucional.

2. **Análogo único:** El análisis se basa en un único evento análogo (Niño 2023–2024). La variabilidad entre episodios El Niño puede ser significativa dependiendo de la intensidad (Niño moderado, fuerte, muy fuerte), la duración y los patrones de teleconexión específicos de cada evento.

3. **Cambio en la exposición y vulnerabilidad:** Las condiciones de exposición y vulnerabilidad de los territorios cambian con el tiempo. Los valores de impacto del análogo 2023–2024 no incorporan los cambios en el ordenamiento territorial, la infraestructura o las dinámicas poblacionales que podrían ocurrir entre 2024 y 2027.

4. **Proyección puntual:** Este análisis no representa una proyección probabilística formal. Para una estimación cuantitativa del riesgo se requiere complementar con modelos probabilísticos de amenaza hidrometeorológica y análisis de vulnerabilidad.

**Recomendaciones para extender el análisis:**

- Incorporar datos de múltiples eventos El Niño (e.g., 2009–2010, 2015–2016, 2023–2024) para construir un ensamble de escenarios.
- Integrar índices climáticos como el Índice Oceánico El Niño (ONI) para correlacionar la magnitud del fenómeno con la intensidad de los impactos.
- Complementar con datos de vulnerabilidad social (Censo DANE 2018) para identificar los municipios con mayor riesgo compuesto (alta amenaza + alta vulnerabilidad).
- Aplicar el mismo flujo de análisis a nivel municipal para los departamentos priorizados en el ranking departamental.

## Referencias

- Unidad Nacional para la Gestión del Riesgo de Desastres (UNGRD). (2024). *Reportes de emergencias históricos — Subdirección para el Conocimiento del Riesgo.* Bogotá: UNGRD.

- Instituto de Hidrología, Meteorología y Estudios Ambientales (IDEAM). (2024). *Boletín de alertas hidrometeorológicas.* Bogotá: IDEAM.

- Departamento Administrativo Nacional de Estadística (DANE). (2023). *División Político-Administrativa de Colombia — DIVIPOLA.* Bogotá: DANE.

- NOAA / Climate Prediction Center. (2024). *El Niño/Southern Oscillation (ENSO) Diagnostic Discussion.* National Weather Service.

- UNDRR. (2015). *Marco de Sendai para la Reducción del Riesgo de Desastres 2015–2030.* Ginebra: Naciones Unidas.

**Fuentes de datos y bibliotecas:**

- Pandas — análisis y manipulación de datos tabulares.
- Altair — visualización declarativa de gráficos estadísticos.
- Folium — mapas interactivos basados en Leaflet.js.
- MyST Markdown / Jupyter — cuadernos reproducibles y publicación científica.

---

**Elaborado por:** Christian Euscátegui — Subdirección para el Conocimiento del Riesgo (SCR/UNGRD)  
**Fecha:** Mayo de 2026  
**Licencia:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)